In [ ]:
#trainng with lora

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from transformers import CLIPModel, CLIPProcessor
from torch.utils.data import DataLoader, Subset
from peft import get_peft_model, LoraConfig, TaskType
from torchattacks import PGD
from collections import defaultdict
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/DataSets2/'

In [ ]:
##transformation

import torchvision.transforms as transforms

pre_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],std=[0.26862954, 0.26130258, 0.27577711])
])

In [ ]:
##dataset downloading
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

tr_dataset=datasets.CIFAR10(root=path, train=True, download=True, transform=pre_transform)
test_dataset=datasets.CIFAR10(root=path, train=False, download=True, transform=pre_transform)

train_size=int(0.9 * len(tr_dataset))
val_size=len(tr_dataset)-train_size
train_dataset, val_dataset=random_split(tr_dataset, [train_size, val_size])

#DataLoaders
train_loader=DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
##getting a mini train data
def balanced_data(dataset, target_size):
    num_classes=10
    per_class=target_size//num_classes
    class_indices=defaultdict(list)

    for idx, (_, label) in enumerate(dataset):
        class_indices[label].append(idx)

    selected_indices=[]
    for cls in range(num_classes):
        selected_indices.extend(random.sample(class_indices[cls], per_class))

    random.shuffle(selected_indices)
    return Subset(dataset, selected_indices)

In [ ]:
train_dataset2=balanced_data(train_dataset, 1000)
val_dataset2=balanced_data(val_dataset,200)
test_dataset2=balanced_data(test_dataset,200)
train_loader2=DataLoader(train_dataset2, batch_size=64, shuffle=True)
val_loader2=DataLoader(val_dataset2, batch_size=64, shuffle=False)
test_loader2=DataLoader(test_dataset2, batch_size=64, shuffle=False)

model SetUp

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

##clip model and its processor from hugging face
import torch
from transformers import CLIPProcessor, CLIPModel

clip_model_name="openai/clip-vit-base-patch32"
##pretrained weights
clip_model=CLIPModel.from_pretrained(clip_model_name)
##clip processor 
clip_processor=CLIPProcessor.from_pretrained(clip_model_name)

#Freeze all original parameters to train only LoRA layers
for param in clip_model.parameters():
    param.requires_grad = False

##LoRa
lora_config=LoraConfig(r=8,lora_alpha=32,target_modules=["q_proj", "v_proj"],  
    lora_dropout=0.1,bias="none",task_type=TaskType.FEATURE_EXTRACTION)
    

clip_model=get_peft_model(clip_model, lora_config)

clip_model=clip_model.to(device)

##Resnet 20
target_model=torch.hub.load("chenyaofo/pytorch-cifar-models","cifar10_resnet20",pretrained=True)
target_model=target_model.to(device)
target_model.eval()

## making text vector
c10_classes=tr_dataset.classes
text_prompts=[f"a photo of a {label}" for label in c10_classes]
text_inputs=clip_processor(text=[f"a photo of a {c}" for c in c10_classes], return_tensors="pt", padding=True).to(device)
text_features=clip_model.get_text_features(**text_inputs)
text_features=F.normalize(text_features, dim=-1)
text_features=text_features.detach()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar10_resnet20-4118986f.pt" to /root/.cache/torch/hub/checkpoints/cifar10_resnet20-4118986f.pt
100%|██████████| 1.09M/1.09M [00:00<00:00, 19.4MB/s

In [ ]:
##optimizer,loss and attack
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, clip_model.parameters()), lr=1e-4)
criterion=nn.CrossEntropyLoss()
pgd=PGD(model=target_model, eps=8/255, alpha=2/255, steps=7)


Making ADV datsets

In [ ]:
def make_adv_dataset(dataloader, pgd_attack, data_name, save_dir, device,batch_size=64 ):
    os.makedirs(save_dir, exist_ok=True)

    adv_images, adv_labels = [], []

    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        adv_imgs = pgd_attack(imgs, labels)
        adv_images.append(adv_imgs.cpu())
        adv_labels.append(labels.cpu())

    adv_images = torch.cat(adv_images)
    adv_labels = torch.cat(adv_labels)

    torch.save(adv_images, os.path.join(save_dir, f"{data_name}_images.pt"))
    torch.save(adv_labels, os.path.join(save_dir, f"{data_name}_labels.pt"))
    dataset = torch.utils.data.TensorDataset(adv_images, adv_labels)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=(data_name == "train"))


In [ ]:
import os
save_dir_data = "/content/drive/MyDrive/clip_lora_adv_datset"

In [16]:
adv_train_loader = make_adv_dataset(train_loader2, pgd, "train", save_dir=save_dir_data, device=device, batch_size=64)

In [17]:
adv_val_loader=make_adv_dataset(val_loader2, pgd, "val", save_dir=save_dir_data , device=device, batch_size=64)

In [18]:
adv_test_loader=make_adv_dataset(test_loader2, pgd, "test", save_dir=save_dir_data , device=device, batch_size=64)

#Training

In [19]:
import os
save_dir = "/content/drive/MyDrive/clip_lora_checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
import time
import os

# Training loop
for epoch in range(15):
    clip_model.train()
    s_time=time.time()

    tr_loss=0.0
    tr_correct=0
    tr_total=0

    for imgs, labels in adv_train_loader:
        imgs, labels=imgs.to(device), labels.to(device)


        pil_imgs=[transforms.ToPILImage()(img.cpu()) for img in imgs]
        inputs=clip_processor(images=pil_imgs, return_tensors="pt")
        inputs={k: v.to(device) for k, v in inputs.items()}
        img_features=clip_model.get_image_features(**inputs)
        img_features=F.normalize(img_features, dim=-1)

        # dot product
        logits=img_features @ text_features.T
        loss=criterion(logits, labels)

        #Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss+=loss.item()
        preds=logits.argmax(dim=1)
        tr_correct+=(preds == labels).sum().item()
        tr_total+=labels.size(0)

    avg_tr_loss=tr_loss/len(adv_train_loader)
    tr_accuracy=100 * tr_correct / tr_total

    # Validation
    clip_model.eval()
    val_loss=0.0
    val_correct=0
    val_total=0


    for val_imgs, val_labels in adv_val_loader:
        val_imgs, val_labels=val_imgs.to(device), val_labels.to(device)


        with torch.no_grad():
            pil_imgs=[transforms.ToPILImage()(img.cpu()) for img in val_imgs]
            inputs=clip_processor(images=pil_imgs, return_tensors="pt")
            inputs={k: v.to(device) for k, v in inputs.items()}
            img_features=clip_model.get_image_features(**inputs)
            img_features=F.normalize(img_features, dim=-1)

            logits=img_features @ text_features.T
            loss=criterion(logits, val_labels)

            val_loss+=loss.item()
            preds=logits.argmax(dim=1)
            val_correct+=(preds == val_labels).sum().item()
            val_total+=val_labels.size(0)

    avg_val_loss=val_loss/len(adv_val_loader)
    val_accuracy=100 * val_correct/val_total

    # Print epoch summary
    end_time=time.time()
    t=(end_time - s_time)/ 60
    print(f"Epoch {epoch+1} takes {t:.2f} min")
    print(f"Training Loss: {avg_tr_loss:.4f}, Training Accuracy: {tr_accuracy:.2f}%")
    print(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

    #Save model for each epoch
    save_path = os.path.join(save_dir, f"clip_lora_epoch{epoch+1}.pt")
    torch.save(clip_model.state_dict(), save_path)

Epoch 1 takes 6.51 min
Training Loss: 2.2588, Training Accuracy: 69.90%
Validation Loss: 2.2420, Validation Accuracy: 81.50%
Epoch 2 takes 6.65 min
Training Loss: 2.2221, Training Accuracy: 78.80%
Validation Loss: 2.1976, Validation Accuracy: 82.00%
Epoch 3 takes 6.77 min
Training Loss: 2.1743, Training Accuracy: 83.10%
Validation Loss: 2.1565, Validation Accuracy: 81.50%
Epoch 4 takes 6.66 min
Training Loss: 2.1363, Training Accuracy: 87.60%
Validation Loss: 2.1315, Validation Accuracy: 82.50%
Epoch 5 takes 6.76 min
Training Loss: 2.1097, Training Accuracy: 90.20%
Validation Loss: 2.1084, Validation Accuracy: 86.50%
Epoch 6 takes 6.73 min
Training Loss: 2.0894, Training Accuracy: 91.80%
Validation Loss: 2.0982, Validation Accuracy: 83.50%
Epoch 7 takes 6.83 min
Training Loss: 2.0732, Training Accuracy: 92.80%
Validation Loss: 2.0864, Validation Accuracy: 85.00%
Epoch 8 takes 6.82 min
Training Loss: 2.0594, Training Accuracy: 94.60%
Validation Loss: 2.0737, Validation Accuracy: 85.50%


In [ ]:
def evaluate(loader):
    clip_model.eval()  
    correct=0
    total=0

    for imgs, labels in loader:

          imgs, labels = imgs.to(device), labels.to(device)

          with torch.no_grad():

            #Converting tensors to list of PIL images for CLIPProcessor
            pil_imgs=[transforms.ToPILImage()(img.cpu()) for img in imgs]

            inputs=clip_processor(images=pil_imgs, return_tensors="pt", padding=True)
            inputs={k: v.to(device) for k, v in inputs.items()}

            img_features=clip_model.get_image_features(**inputs)
            img_features=F.normalize(img_features, dim=-1)

            logits=img_features @ text_features.T
            preds=logits.argmax(dim=1)

            
            correct+=(preds == labels).sum().item()
            total+=labels.size(0)

    accuracy_percent = 100 * correct / total
    return accuracy_percent

In [22]:
clean_acc=evaluate(test_loader2)

In [23]:
print(f"Clean Accuracy: {clean_acc:.2f}%")

Clean Accuracy: 46.00%


In [24]:
adv_acc=evaluate(adv_test_loader)

In [25]:
print(f"Adversarial Accuracy: {adv_acc:.2f}%")

Adversarial Accuracy: 84.50%
